# Driving McIDAS-V with Python-computed parameters

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## Compute display parameters in numpy / pandas

In [ ]:
import numpy as np
import pandas as pd

sample = 210 + 90 * np.random.rand(5000)         # 'brightness temps'
vmin, vmax = np.percentile(sample, [2, 98]).round(1).tolist()
levels = np.linspace(vmin, vmax, 9).round(1).tolist()
center_lat = float(np.average([30, 45], weights=[1, 3]))
positions = list(range(-4, 1))
print('range:', (vmin, vmax))
print('levels:', levels)
print('center_lat:', center_lat)

## Use them in a McIDAS-V script with `--push`

In [ ]:
%%mcv --push vmin,vmax,center_lat,positions
adde = dict(server='adde.ucar.edu', dataset='EAST',
            descriptor='CONUSC13', size='ALL', unit='TEMP')

frames = [loadADDEImage(position=p, **adde) for p in positions]

panel = buildWindow(height=600, width=800)
layer = panel[0].createLayer('Image Sequence Display', frames)
panel[0].setWireframe(False)
panel[0].setCenter(center_lat, -95.0, scale=1.0)

layer.setEnhancement('ABI IR Temperature', range=(vmin, vmax))

## The same thing via the object API

In [ ]:
import mcidasv_jupyter as mcv
session = mcv.get_session()

result = session.run('''
print 'contour levels from numpy:', levels
print 'that is %d levels between %g and %g' % (len(levels), levels[0], levels[-1])
''', values={'levels': levels}, capture=None, display=False)
print(result.prints)